# MeditActive — Sistema di Gestione degli Elementi

**Fabio Mencio | Progetto Python**

MeditActive permette di registrare, cercare e completare obiettivi di benessere.
Ogni sessione completata genera monete virtuali e ogni 50 monete viene simulata
la piantumazione di un albero.

Il progetto è costruito con le strutture di base del linguaggio: liste,
dizionari, funzioni, cicli `for` e `while`, condizioni `if/else`, parametri e
valori restituiti con `return`.

## 1. Come eseguire e leggere il notebook

Aprire il file in Colab o Jupyter ed eseguire tutte le celle dall'alto verso il
basso. Non servono altri file né pacchetti da installare. Per ripartire dai dati
iniziali, riavviare il kernel ed eseguire nuovamente tutto.

La collezione è una **lista di dizionari**. Il saldo è un numero intero.

Tre scelte guidano tutto il codice che segue.

**Le funzioni ricevono nei parametri tutto ciò che usano.** Nessuna funzione
legge o modifica una variabile della dimostrazione: niente `global`. Una lista
passata come parametro può essere aggiornata dalla funzione che la riceve; per
il saldo, invece, la funzione restituisce un nuovo numero e il programma lo
riassegna:

```python
saldo_monete = completa_obiettivo(collezione_obiettivi, 4, saldo_monete, adesso)
```

**L'istante di riferimento è un parametro, non l'orologio di sistema.** Le
funzioni che devono sapere "che giorno è" ricevono `adesso`. Questo rende il
comportamento verificabile: per mostrare cosa succede una settimana dopo basta
passare un'altra data, senza aspettare una settimana.

**Un dato che si può calcolare non viene memorizzato.** Le monete di una
sessione dipendono dalla durata, e lo stato "completato" dipende dall'ultima
sessione e dalla frequenza: entrambi si ricavano quando servono, così non
possono entrare in contraddizione con i dati da cui derivano.

Non ci sono classi, funzioni `lambda`, comprensioni di liste o eccezioni.
L'unica importazione è `datetime`, per le date e per il calcolo dei periodi.
Gli input non validi sono gestiti con `if`, un messaggio e `return`.

## 2. Regole del dominio e costanti condivise

Le categorie ammesse, le frequenze e le regole di conversione sono **costanti
scritte una volta sola**, fuori dalle funzioni. Sono valori di sola lettura: ogni
funzione che deve validare un dato guarda la stessa lista, quindi la
registrazione e la ricerca non possono divergere.

| Costante | Contenuto |
|---|---|
| `TIPI_AMMESSI` | le quattro categorie di attività |
| `FREQUENZE_AMMESSE` | le quattro periodicità previste |
| `GIORNI_PERIODO` | quanti giorni dura un periodo per ciascuna frequenza |
| `MINUTI_PER_MONETA` | minuti di pratica che valgono una moneta |
| `MONETE_PER_ALBERO` | monete necessarie per un albero |

Tre funzioni brevi traducono queste regole in risposte.

`monete_per_sessione` converte la durata in monete: 12 minuti valgono 2 monete,
da 1 a 4 minuti valgono 0. La divisione `//` tiene solo la parte intera.

`completato_nel_periodo` dice se l'obiettivo è già stato completato nel periodo
in corso. Sottraendo due date si ottiene la distanza fra loro, e `.days` ne legge
i giorni interi.

`testo_data` formatta una data per la stampa e scrive `mai` quando la data non
c'è ancora.

In [1]:
from datetime import datetime, timedelta

TIPI_AMMESSI = ["Meditazione", "Movimento", "Introspezione", "Sonno"]
FREQUENZE_AMMESSE = ["Giornaliera", "Settimanale", "Mensile", "Annuale"]

GIORNI_PERIODO = {
    "Giornaliera": 1,
    "Settimanale": 7,
    "Mensile": 30,
    "Annuale": 365,
}

MINUTI_PER_MONETA = 5
MONETE_PER_ALBERO = 50


def monete_per_sessione(durata_minuti):
    """Converti la durata di una sessione nelle monete guadagnate.

    Parametri:
        durata_minuti (int): Durata della sessione in minuti.

    Restituisce:
        int: Una moneta ogni MINUTI_PER_MONETA minuti interi.
    """
    return durata_minuti // MINUTI_PER_MONETA


def completato_nel_periodo(elemento, adesso):
    """Indica se l'obiettivo è già stato completato nel periodo in corso.

    Parametri:
        elemento (dict): Obiettivo da esaminare, non viene modificato.
        adesso (datetime): Istante di riferimento della valutazione.

    Restituisce:
        bool: True se l'ultima sessione ricade nel periodo corrente.
        False se l'obiettivo non è mai stato completato, oppure se il
        periodo previsto dalla sua frequenza si è già concluso.
    """
    if elemento["ultima_sessione"] is None:
        return False

    giorni_trascorsi = (adesso - elemento["ultima_sessione"]).days
    return giorni_trascorsi < GIORNI_PERIODO[elemento["frequenza"]]


def testo_data(momento):
    """Prepara una data per la stampa.

    Parametri:
        momento (datetime o None): Data da formattare.

    Restituisce:
        str: La data come giorno/mese/anno e ora, oppure "mai" se non c'è.
    """
    if momento is None:
        return "mai"

    return momento.strftime("%d/%m/%Y %H:%M")

## 3. Registrazione e visualizzazione

La registrazione controlla, nell'ordine:

1. Nome, tipo e frequenza devono essere testi non vuoti.
2. La categoria deve essere una di `TIPI_AMMESSI`.
3. La frequenza deve essere una di `FREQUENZE_AMMESSE`.
4. La durata deve essere un intero positivo: `"trenta"`, `"30"`, `30.0`, `True`,
   zero e i numeri negativi non sono validi.
5. Il nome non deve essere già presente, anche con maiuscole diverse.

Per i testi si usa `isinstance(nome, str)`, la forma idiomatica in Python. Per la
durata serve invece `type(durata_minuti) != int`: in Python `True` è anche un
intero, e `isinstance(True, int)` risponde `True`. Solo il confronto diretto del
tipo riesce a rifiutarlo.

`strip()` elimina gli spazi all'inizio e alla fine. `lower()` permette di
confrontare i nomi senza distinguere maiuscole e minuscole.

Un ciclo cerca l'ID più alto e aggiunge uno, così un identificativo non viene
riassegnato dopo una rimozione. `zfill(3)` aggiunge gli zeri iniziali: `"4"`
diventa `"004"`, e gli ID oltre 999 non vengono troncati.

In [2]:
def registrazione_nuovo_elemento(
    collezione, nome, tipo, frequenza, durata_minuti, adesso
):
    """Controlla i dati e aggiungi un nuovo obiettivo alla collezione.

    Parametri:
        collezione (list): Lista degli obiettivi da aggiornare.
        nome (str): Nome non vuoto e non già presente nella collezione.
        tipo (str): Categoria fra quelle elencate in TIPI_AMMESSI.
        frequenza (str): Periodicità fra quelle in FREQUENZE_AMMESSE.
        durata_minuti (int): Durata intera maggiore di zero.
        adesso (datetime): Istante da registrare come data di inserimento.

    Restituisce:
        bool: True se l'inserimento riesce, False se i dati non sono validi.
        In caso di errore stampa un messaggio e lascia la collezione invariata.
    """
    if not isinstance(nome, str):
        print("Errore: il nome deve essere un testo.")
        return False

    if not isinstance(tipo, str) or not isinstance(frequenza, str):
        print("Errore: tipo e frequenza devono essere testi.")
        return False

    nome = nome.strip()
    tipo = tipo.strip().capitalize()
    frequenza = frequenza.strip().capitalize()

    if nome == "" or tipo == "" or frequenza == "":
        print("Errore: nome, tipo e frequenza non possono essere vuoti.")
        return False

    if tipo not in TIPI_AMMESSI:
        print(f"Errore: categoria '{tipo}' non valida.")
        print(f"Categorie ammesse: {', '.join(TIPI_AMMESSI)}.")
        return False

    if frequenza not in FREQUENZE_AMMESSE:
        print(f"Errore: frequenza '{frequenza}' non valida.")
        print(f"Frequenze ammesse: {', '.join(FREQUENZE_AMMESSE)}.")
        return False

    # Qui serve type(): isinstance(True, int) risponde True e lascerebbe
    # passare un valore booleano al posto di una durata.
    if type(durata_minuti) != int:
        print("Errore: la durata deve essere un numero intero.")
        return False

    if durata_minuti <= 0:
        print("Errore: la durata deve essere maggiore di zero.")
        return False

    for elemento in collezione:
        if elemento["nome"].lower() == nome.lower():
            print("Errore: esiste già un obiettivo con questo nome.")
            return False

    # Il nuovo ID è il massimo presente più uno, non len() + 1: così una
    # rimozione non fa riapparire un identificativo già usato.
    id_massimo = 0
    for elemento in collezione:
        id_numero = int(elemento["ID"])
        if id_numero > id_massimo:
            id_massimo = id_numero

    nuovo_id = str(id_massimo + 1).zfill(3)

    nuovo_obiettivo = {
        "ID": nuovo_id,
        "nome": nome,
        "tipo": tipo,
        "frequenza": frequenza,
        "durata_minuti": durata_minuti,
        "volte_completato": 0,
        "ultima_sessione": None,
        "data_inserimento": adesso,
    }

    collezione.append(nuovo_obiettivo)
    print(f"Obiettivo aggiunto: [{nuovo_id}] {nome}")
    return True

La visualizzazione mostra ogni obiettivo in un piccolo blocco di testo. La
stessa funzione serve per il dataset, per i nuovi obiettivi e per i risultati
delle ricerche. Se la lista è vuota, il messaggio lo dice esplicitamente.

Le monete e lo stato non vengono letti dal dizionario: sono calcolati al momento
della stampa, dalla durata e dall'ultima sessione.

In [3]:
def visualizza_collezione(collezione, adesso):
    """Mostra gli obiettivi in un formato leggibile.

    Parametri:
        collezione (list): Lista completa oppure risultati di una ricerca.
        adesso (datetime): Istante usato per dire se il periodo è ancora aperto.

    Restituisce:
        None: Stampa gli obiettivi, oppure un messaggio se la lista è vuota.
        Non modifica i dati.
    """
    if len(collezione) == 0:
        print("La collezione è vuota: nessun obiettivo da mostrare.")
        return

    for elemento in collezione:
        if completato_nel_periodo(elemento, adesso):
            stato = "Completato in questo periodo"
        else:
            stato = "Da completare"

        monete = monete_per_sessione(elemento["durata_minuti"])

        print(f"\n[{elemento['ID']}] {elemento['nome']}")
        print(f"  Tipo: {elemento['tipo']} | Frequenza: {elemento['frequenza']}")
        print(f"  Durata: {elemento['durata_minuti']} minuti")
        print(f"  Monete per sessione: {monete}")
        print(f"  Stato: {stato} | Sessioni: {elemento['volte_completato']}")
        print(f"  Ultima sessione: {testo_data(elemento['ultima_sessione'])}")
        print(f"  Inserito il: {testo_data(elemento['data_inserimento'])}")

## 4. Dataset iniziale

La funzione seguente restituisce i dieci obiettivi di partenza. Ogni chiamata
crea una lista nuova, utile per tenere le prove separate dalla dimostrazione
principale.

Tutti gli elementi hanno le stesse chiavi:

| Chiave | Significato |
|---|---|
| `ID` | identificativo dell'obiettivo |
| `nome` | nome dell'attività |
| `tipo` | categoria dell'attività |
| `frequenza` | periodicità prevista |
| `durata_minuti` | durata di una sessione |
| `volte_completato` | numero di sessioni registrate |
| `ultima_sessione` | data dell'ultima sessione, `None` se mai completato |
| `data_inserimento` | data di aggiunta alla collezione |

Le monete e lo stato di completamento **non** sono chiavi del dizionario: si
ricavano dalla durata e dall'ultima sessione. Un campo memorizzato sarebbe una
seconda verità da tenere allineata, e prima o poi divergerebbe — se un giorno la
durata cambiasse, le monete salvate resterebbero quelle vecchie.

Le date e le ore sono oggetti `datetime`, non testo. Si confrontano e si
sommano direttamente, senza dipendere dal formato con cui vengono scritte; la
formattazione avviene solo al momento della stampa.

Le date dal 1 al 10 settembre 2026 sono **fittizie**, servono al dataset di
esempio. Ogni nuovo inserimento riceve invece l'istante passato in `adesso`.

In [4]:
def crea_collezione_iniziale():
    """Crea una nuova lista con i dieci obiettivi iniziali.

    Parametri:
        Nessuno.

    Restituisce:
        list: Dieci dizionari con i contatori a zero, nessuna sessione
        registrata e date di inserimento fittizie.
    """
    collezione = [
        {
            "ID": "001",
            "nome": "Meditazione guidata mattutina",
            "tipo": "Meditazione",
            "frequenza": "Giornaliera",
            "durata_minuti": 10,
            "volte_completato": 0,
            "ultima_sessione": None,
            "data_inserimento": datetime(2026, 9, 1, 9, 0),
        },
        {
            "ID": "002",
            "nome": "Camminata consapevole nel bosco",
            "tipo": "Movimento",
            "frequenza": "Giornaliera",
            "durata_minuti": 30,
            "volte_completato": 0,
            "ultima_sessione": None,
            "data_inserimento": datetime(2026, 9, 2, 9, 0),
        },
        {
            "ID": "003",
            "nome": "Diario della gratitudine serale",
            "tipo": "Introspezione",
            "frequenza": "Giornaliera",
            "durata_minuti": 15,
            "volte_completato": 0,
            "ultima_sessione": None,
            "data_inserimento": datetime(2026, 9, 3, 9, 0),
        },
        {
            "ID": "004",
            "nome": "Routine pre sonno",
            "tipo": "Sonno",
            "frequenza": "Giornaliera",
            "durata_minuti": 20,
            "volte_completato": 0,
            "ultima_sessione": None,
            "data_inserimento": datetime(2026, 9, 4, 9, 0),
        },
        {
            "ID": "005",
            "nome": "Sfida respirazione 4-7-8",
            "tipo": "Meditazione",
            "frequenza": "Mensile",
            "durata_minuti": 25,
            "volte_completato": 0,
            "ultima_sessione": None,
            "data_inserimento": datetime(2026, 9, 5, 9, 0),
        },
        {
            "ID": "006",
            "nome": "Digital detox del week-end",
            "tipo": "Introspezione",
            "frequenza": "Mensile",
            "durata_minuti": 120,
            "volte_completato": 0,
            "ultima_sessione": None,
            "data_inserimento": datetime(2026, 9, 6, 9, 0),
        },
        {
            "ID": "007",
            "nome": "Percorso base body scan",
            "tipo": "Meditazione",
            "frequenza": "Mensile",
            "durata_minuti": 45,
            "volte_completato": 0,
            "ultima_sessione": None,
            "data_inserimento": datetime(2026, 9, 7, 9, 0),
        },
        {
            "ID": "008",
            "nome": "Mobilità dolce, yoga e zen",
            "tipo": "Movimento",
            "frequenza": "Mensile",
            "durata_minuti": 50,
            "volte_completato": 0,
            "ultima_sessione": None,
            "data_inserimento": datetime(2026, 9, 8, 9, 0),
        },
        {
            "ID": "009",
            "nome": "Masterclass gestione dello stress",
            "tipo": "Introspezione",
            "frequenza": "Annuale",
            "durata_minuti": 60,
            "volte_completato": 0,
            "ultima_sessione": None,
            "data_inserimento": datetime(2026, 9, 9, 9, 0),
        },
        {
            "ID": "010",
            "nome": "Consapevolezza indoor",
            "tipo": "Meditazione",
            "frequenza": "Annuale",
            "durata_minuti": 180,
            "volte_completato": 0,
            "ultima_sessione": None,
            "data_inserimento": datetime(2026, 9, 10, 9, 0),
        },
    ]
    return collezione

In [5]:
adesso = datetime.now()
collezione_obiettivi = crea_collezione_iniziale()
saldo_monete = 0

visualizza_collezione(collezione_obiettivi, adesso)


[001] Meditazione guidata mattutina
  Tipo: Meditazione | Frequenza: Giornaliera
  Durata: 10 minuti
  Monete per sessione: 2
  Stato: Da completare | Sessioni: 0
  Ultima sessione: mai
  Inserito il: 01/09/2026 09:00

[002] Camminata consapevole nel bosco
  Tipo: Movimento | Frequenza: Giornaliera
  Durata: 30 minuti
  Monete per sessione: 6
  Stato: Da completare | Sessioni: 0
  Ultima sessione: mai
  Inserito il: 02/09/2026 09:00

[003] Diario della gratitudine serale
  Tipo: Introspezione | Frequenza: Giornaliera
  Durata: 15 minuti
  Monete per sessione: 3
  Stato: Da completare | Sessioni: 0
  Ultima sessione: mai
  Inserito il: 03/09/2026 09:00

[004] Routine pre sonno
  Tipo: Sonno | Frequenza: Giornaliera
  Durata: 20 minuti
  Monete per sessione: 4
  Stato: Da completare | Sessioni: 0
  Ultima sessione: mai
  Inserito il: 04/09/2026 09:00

[005] Sfida respirazione 4-7-8
  Tipo: Meditazione | Frequenza: Mensile
  Durata: 25 minuti
  Monete per sessione: 5
  Stato: Da completa

## 5. Inserimento di due nuovi obiettivi

La funzione restituisce `True` oppure `False`, così il nuovo elemento viene
mostrato soltanto se l'inserimento è riuscito. `collezione_obiettivi[-1]` indica
l'ultimo obiettivo: viene passato alla visualizzazione dentro una lista, per non
stampare il dizionario grezzo.

In [6]:
inserito = registrazione_nuovo_elemento(
    collezione_obiettivi,
    "Respirazione consapevole al tramonto", "Meditazione", "Giornaliera", 15,
    adesso
)
if inserito:
    visualizza_collezione([collezione_obiettivi[-1]], adesso)

inserito = registrazione_nuovo_elemento(
    collezione_obiettivi,
    "Seduta dal terapeuta", "Introspezione", "Settimanale", 60,
    adesso
)
if inserito:
    visualizza_collezione([collezione_obiettivi[-1]], adesso)

Obiettivo aggiunto: [011] Respirazione consapevole al tramonto

[011] Respirazione consapevole al tramonto
  Tipo: Meditazione | Frequenza: Giornaliera
  Durata: 15 minuti
  Monete per sessione: 3
  Stato: Da completare | Sessioni: 0
  Ultima sessione: mai
  Inserito il: 24/09/2026 17:13
Obiettivo aggiunto: [012] Seduta dal terapeuta

[012] Seduta dal terapeuta
  Tipo: Introspezione | Frequenza: Settimanale
  Durata: 60 minuti
  Monete per sessione: 12
  Stato: Da completare | Sessioni: 0
  Ultima sessione: mai
  Inserito il: 24/09/2026 17:13


## 6. Completamento, frequenza, monete e alberi

La funzione accetta un ID oppure il **nome completo**, ignorando le maiuscole.
`str()` converte l'identificativo in testo: anche il numero `4` trova
l'obiettivo `004`. Per cercare con una parte del nome si usa il filtro della
sezione 7.

**La frequenza è una regola, non un'etichetta.** Un obiettivo giornaliero vale
una sessione al giorno; uno mensile, una ogni trenta giorni. Prima di registrare
una sessione la funzione chiede a `completato_nel_periodo` se il periodo in
corso è già stato usato: se lo è, rifiuta il completamento, non assegna monete e
dice da quando la prossima sessione sarà valida. Quando il periodo scade,
l'obiettivo torna da solo fra quelli da completare — non serve azzerare nulla,
perché lo stato è calcolato e non memorizzato.

Le monete si ricavano dalla durata al momento del completamento: cinque minuti
interi valgono una moneta, quindi da 1 a 4 minuti valgono 0. Il ciclo `while`
sottrae 50 monete per ciascun albero simulato. Non si effettuano acquisti né
piantumazioni reali.

In [7]:
def completa_obiettivo(collezione, identificativo, saldo_monete, adesso):
    """Registra una sessione, se la frequenza dell'obiettivo lo consente.

    Parametri:
        collezione (list): Lista degli obiettivi da aggiornare.
        identificativo (str o int): ID oppure nome completo dell'obiettivo.
            Il nome ignora le maiuscole e gli spazi iniziali e finali.
        saldo_monete (int): Saldo disponibile prima della sessione.
        adesso (datetime): Istante in cui la sessione viene registrata.

    Restituisce:
        int: Nuovo saldo dopo il guadagno e gli eventuali alberi simulati.
        Restituisce il saldo ricevuto, invariato, se l'obiettivo non esiste
        oppure se è già stato completato nel periodo in corso.
        Quando la sessione è valida aggiorna il contatore e l'ultima data
        dell'obiettivo trovato, e stampa l'esito.
    """
    testo = str(identificativo).strip().lower()
    id_cercato = testo.zfill(3)
    trovato = None

    for elemento in collezione:
        if elemento["ID"] == id_cercato or elemento["nome"].lower() == testo:
            trovato = elemento
            break

    if trovato is None:
        print("Nessun obiettivo trovato con questo ID o nome.\n")
        return saldo_monete

    if completato_nel_periodo(trovato, adesso):
        giorni = GIORNI_PERIODO[trovato["frequenza"]]
        prossima = trovato["ultima_sessione"] + timedelta(days=giorni)

        if giorni == 1:
            cadenza = "una sessione al giorno"
        else:
            cadenza = f"una sessione ogni {giorni} giorni"

        print(f"Già completato: [{trovato['ID']}] {trovato['nome']}")
        print(f"Frequenza {trovato['frequenza'].lower()}: {cadenza}.")
        print(f"Ultima sessione: {testo_data(trovato['ultima_sessione'])}")
        print(f"Prossima sessione valida dal {testo_data(prossima)}.\n")
        return saldo_monete

    monete = monete_per_sessione(trovato["durata_minuti"])
    trovato["volte_completato"] += 1
    trovato["ultima_sessione"] = adesso
    saldo_monete += monete

    print(f"Completato: [{trovato['ID']}] {trovato['nome']}")
    print(f"Sessioni registrate: {trovato['volte_completato']}")
    print(f"Monete guadagnate: {monete}")

    while saldo_monete >= MONETE_PER_ALBERO:
        saldo_monete -= MONETE_PER_ALBERO
        print(f"Albero piantato nella simulazione! Costo: {MONETE_PER_ALBERO} monete.")

    print(f"Saldo disponibile: {saldo_monete} monete.\n")
    return saldo_monete

In [8]:
saldo_monete = completa_obiettivo(collezione_obiettivi, 4, saldo_monete, adesso)

# Lo stesso obiettivo, chiamato per nome: è giornaliero e oggi è già stato
# fatto, quindi la sessione viene rifiutata e il saldo non cambia.
saldo_monete = completa_obiettivo(
    collezione_obiettivi, "  ROUTINE PRE SONNO  ", saldo_monete, adesso
)

saldo_monete = completa_obiettivo(collezione_obiettivi, "010", saldo_monete, adesso)
saldo_monete = completa_obiettivo(collezione_obiettivi, "006", saldo_monete, adesso)

print("Atteso: 3 sessioni valide, 64 monete guadagnate, 1 albero, saldo 14.")
print(f"Saldo ottenuto: {saldo_monete}")

Completato: [004] Routine pre sonno
Sessioni registrate: 1
Monete guadagnate: 4
Saldo disponibile: 4 monete.

Già completato: [004] Routine pre sonno
Frequenza giornaliera: una sessione al giorno.
Ultima sessione: 24/09/2026 17:13
Prossima sessione valida dal 25/09/2026 17:13.

Completato: [010] Consapevolezza indoor
Sessioni registrate: 1
Monete guadagnate: 36
Saldo disponibile: 40 monete.

Completato: [006] Digital detox del week-end
Sessioni registrate: 1
Monete guadagnate: 24
Albero piantato nella simulazione! Costo: 50 monete.
Saldo disponibile: 14 monete.

Atteso: 3 sessioni valide, 64 monete guadagnate, 1 albero, saldo 14.
Saldo ottenuto: 14


## 7. Ricerca e filtri

Ogni filtro è facoltativo: `None` significa che quel criterio non viene usato.
I criteri indicati devono essere rispettati tutti.

- `nome`: cerca anche una parte del nome, ignorando le maiuscole.
- `tipo_cercato`: una delle categorie di `TIPI_AMMESSI`, ignorando le maiuscole.
- `durata_massima`: include anche la durata uguale al limite.
- `solo_completati`: con `True` cerca quelli già completati nel periodo in
  corso, con `False` quelli ancora da fare.

I parametri vengono controllati prima del ciclo, anche quando la collezione è
vuota. La variabile `corrisponde` parte da `True` e diventa `False` appena un
criterio non è rispettato.

**La categoria viene validata contro le stesse costanti usate in
registrazione.** Senza questo controllo una ricerca scritta male —
`tipo_cercato="meditaz"` — restituirebbe una lista vuota indistinguibile da "non
ho trovato niente", e chi cerca penserebbe che quegli obiettivi non esistono. Un
filtro non valido stampa un errore e restituisce `[]`.

In [9]:
def filtro_avanzato(
    collezione, adesso, tipo_cercato=None, durata_massima=None,
    solo_completati=None, nome=None
):
    """Cerca gli obiettivi che rispettano tutti i criteri indicati.

    Parametri:
        collezione (list): Lista in cui cercare, non viene modificata.
        adesso (datetime): Istante usato per valutare lo stato di
            completamento quando serve il filtro solo_completati.
        tipo_cercato (str o None): Categoria fra quelle di TIPI_AMMESSI,
            ignorando maiuscole e spazi.
        durata_massima (int o None): Durata massima positiva, inclusa.
        solo_completati (bool o None): True cerca quelli completati nel
            periodo in corso, False quelli ancora da completare.
        nome (str o None): Parte del nome, ignorando le maiuscole.
            Ogni parametro con valore None disattiva il relativo filtro.

    Restituisce:
        list: Obiettivi trovati, nell'ordine della collezione.
        Restituisce [] se non trova risultati o se un filtro non è valido;
        nel secondo caso stampa anche un messaggio di errore.
    """
    # I parametri si controllano prima di scorrere la lista.
    if tipo_cercato is not None:
        if not isinstance(tipo_cercato, str):
            print("Errore: il filtro tipo deve essere un testo.")
            return []
        tipo_cercato = tipo_cercato.strip().capitalize()
        if tipo_cercato == "":
            print("Errore: il filtro tipo non può essere vuoto.")
            return []
        if tipo_cercato not in TIPI_AMMESSI:
            print(f"Errore: categoria '{tipo_cercato}' non valida.")
            print(f"Categorie ammesse: {', '.join(TIPI_AMMESSI)}.")
            return []

    if nome is not None:
        if not isinstance(nome, str):
            print("Errore: il filtro nome deve essere un testo.")
            return []
        nome = nome.strip().lower()
        if nome == "":
            print("Errore: il filtro nome non può essere vuoto.")
            return []

    if durata_massima is not None:
        if type(durata_massima) != int:
            print("Errore: la durata massima deve essere un numero intero.")
            return []
        if durata_massima <= 0:
            print("Errore: la durata massima deve essere maggiore di zero.")
            return []

    if solo_completati is not None:
        if not isinstance(solo_completati, bool):
            print("Errore: solo_completati deve essere True, False o None.")
            return []

    risultati = []
    for elemento in collezione:
        corrisponde = True

        if tipo_cercato is not None:
            if elemento["tipo"] != tipo_cercato:
                corrisponde = False

        if durata_massima is not None:
            if elemento["durata_minuti"] > durata_massima:
                corrisponde = False

        if solo_completati is not None:
            if completato_nel_periodo(elemento, adesso) != solo_completati:
                corrisponde = False

        if nome is not None:
            if nome not in elemento["nome"].lower():
                corrisponde = False

        if corrisponde:
            risultati.append(elemento)

    return risultati

In [10]:
print("RICERCA PER NOME: 'RESPIRAZIONE'")
risultati = filtro_avanzato(collezione_obiettivi, adesso, nome="RESPIRAZIONE")
visualizza_collezione(risultati, adesso)

print("\nMEDITAZIONI FINO A 30 MINUTI, ANCORA DA COMPLETARE")
risultati = filtro_avanzato(
    collezione_obiettivi, adesso, tipo_cercato="meditazione",
    durata_massima=30, solo_completati=False
)
visualizza_collezione(risultati, adesso)

print("\nTUTTI I FILTRI: SONNO, FINO A 20 MINUTI, COMPLETATO, NOME 'PRE'")
risultati = filtro_avanzato(
    collezione_obiettivi, adesso, tipo_cercato="Sonno", durata_massima=20,
    solo_completati=True, nome="PRE"
)
visualizza_collezione(risultati, adesso)

RICERCA PER NOME: 'RESPIRAZIONE'

[005] Sfida respirazione 4-7-8
  Tipo: Meditazione | Frequenza: Mensile
  Durata: 25 minuti
  Monete per sessione: 5
  Stato: Da completare | Sessioni: 0
  Ultima sessione: mai
  Inserito il: 05/09/2026 09:00

[011] Respirazione consapevole al tramonto
  Tipo: Meditazione | Frequenza: Giornaliera
  Durata: 15 minuti
  Monete per sessione: 3
  Stato: Da completare | Sessioni: 0
  Ultima sessione: mai
  Inserito il: 24/09/2026 17:13

MEDITAZIONI FINO A 30 MINUTI, ANCORA DA COMPLETARE

[001] Meditazione guidata mattutina
  Tipo: Meditazione | Frequenza: Giornaliera
  Durata: 10 minuti
  Monete per sessione: 2
  Stato: Da completare | Sessioni: 0
  Ultima sessione: mai
  Inserito il: 01/09/2026 09:00

[005] Sfida respirazione 4-7-8
  Tipo: Meditazione | Frequenza: Mensile
  Durata: 25 minuti
  Monete per sessione: 5
  Stato: Da completare | Sessioni: 0
  Ultima sessione: mai
  Inserito il: 05/09/2026 09:00

[011] Respirazione consapevole al tramonto
  Tipo

## 8. Statistiche

Un ciclo conta gli obiettivi, le sessioni e le categorie. Per trovare il valore
più alto confronta ogni elemento con il massimo incontrato fino a quel momento:
così non servono `lambda` né la sintassi `max(..., key=...)`.

La funzione mostra il tipo più frequente, l'obiettivo più completato e quello
inserito più di recente. Le date sono oggetti `datetime` e si confrontano
direttamente con `>=`, senza dipendere da come vengono scritte.

In caso di parità:

- per il tipo più frequente e per l'obiettivo più completato resta il primo
  incontrato nella collezione;
- per la data più recente viene scelto l'ultimo elemento incontrato, cioè
  l'ultimo aggiunto nell'ordine della lista.

Se non ci sono sessioni, nessun obiettivo viene indicato come più completato. Se
la lista è vuota, la funzione lo dice e si ferma, senza dividere per zero.

In [11]:
def statistiche_elementi(collezione, saldo_monete, adesso):
    """Calcola e mostra le statistiche della collezione.

    Parametri:
        collezione (list): Obiettivi da analizzare, non vengono modificati.
        saldo_monete (int): Saldo disponibile da mostrare nella dashboard.
        adesso (datetime): Istante usato per contare i completamenti validi
            nel periodo in corso.

    Restituisce:
        dict: Conteggi, distribuzione per tipo, tipo più frequente, nome
            dell'obiettivo più completato e nome di quello più recente.
        None: Se la collezione è vuota, dopo un messaggio esplicativo.
        Senza sessioni registrate, il nome dell'obiettivo più completato è None.
    """
    print("DASHBOARD MEDITACTIVE")
    print(f"Saldo monete: {saldo_monete}")

    if len(collezione) == 0:
        print("Nessun dato disponibile: la collezione è vuota.")
        return None

    totale_obiettivi = len(collezione)
    completati_nel_periodo = 0
    sessioni_totali = 0
    distribuzione_tipo = {}
    massimo_completamenti = 0
    nome_piu_completato = None
    piu_recente = collezione[0]

    for elemento in collezione:
        if completato_nel_periodo(elemento, adesso):
            completati_nel_periodo += 1

        sessioni_totali += elemento["volte_completato"]

        tipo = elemento["tipo"]
        if tipo in distribuzione_tipo:
            distribuzione_tipo[tipo] += 1
        else:
            distribuzione_tipo[tipo] = 1

        if elemento["volte_completato"] > massimo_completamenti:
            massimo_completamenti = elemento["volte_completato"]
            nome_piu_completato = elemento["nome"]

        if elemento["data_inserimento"] >= piu_recente["data_inserimento"]:
            piu_recente = elemento

    tipo_piu_frequente = ""
    massimo_tipo = 0
    for tipo in distribuzione_tipo:
        if distribuzione_tipo[tipo] > massimo_tipo:
            massimo_tipo = distribuzione_tipo[tipo]
            tipo_piu_frequente = tipo

    print(f"Totale obiettivi: {totale_obiettivi}")
    print(f"Completati nel periodo in corso: {completati_nel_periodo}")
    print(f"Sessioni totali registrate: {sessioni_totali}")
    print("\nDistribuzione per tipo:")
    for tipo in distribuzione_tipo:
        quantita = distribuzione_tipo[tipo]
        percentuale = round(quantita / totale_obiettivi * 100, 1)
        print(f"- {tipo}: {quantita} ({percentuale}%)")

    print(f"\nTipo più frequente: {tipo_piu_frequente}")
    if nome_piu_completato is None:
        print("Obiettivo più completato: nessuno, non ci sono sessioni.")
    else:
        print(f"Obiettivo più completato: {nome_piu_completato}")
        print(f"Numero di sessioni: {massimo_completamenti}")

    print(f"Obiettivo più recente: {piu_recente['nome']}")
    print(f"Inserito il: {testo_data(piu_recente['data_inserimento'])}")

    riepilogo = {
        "totale_obiettivi": totale_obiettivi,
        "completati_nel_periodo": completati_nel_periodo,
        "sessioni_totali": sessioni_totali,
        "saldo_monete": saldo_monete,
        "distribuzione_tipo": distribuzione_tipo,
        "tipo_piu_frequente": tipo_piu_frequente,
        "obiettivo_piu_completato": nome_piu_completato,
        "obiettivo_piu_recente": piu_recente["nome"],
    }
    return riepilogo

In [12]:
riepilogo = statistiche_elementi(collezione_obiettivi, saldo_monete, adesso)

DASHBOARD MEDITACTIVE
Saldo monete: 14
Totale obiettivi: 12
Completati nel periodo in corso: 3
Sessioni totali registrate: 3

Distribuzione per tipo:
- Meditazione: 5 (41.7%)
- Movimento: 2 (16.7%)
- Introspezione: 4 (33.3%)
- Sonno: 1 (8.3%)

Tipo più frequente: Meditazione
Obiettivo più completato: Routine pre sonno
Numero di sessioni: 1
Obiettivo più recente: Seduta dal terapeuta
Inserito il: 24/09/2026 17:13


## 9. Comportamento con dati non validi

Le prove seguenti usano liste e saldi separati, così la dimostrazione principale
resta invariata. Sono i casi in cui un programma di solito si rompe: lista
vuota, input del tipo sbagliato, nome duplicato, ricerca senza risultati,
identificativo inesistente. Ognuna mostra l'esito e il valore atteso.

In [13]:
print("COLLEZIONE VUOTA")
collezione_vuota = []
visualizza_collezione(collezione_vuota, adesso)
statistiche_vuote = statistiche_elementi(collezione_vuota, 0, adesso)
saldo_vuoto = completa_obiettivo(collezione_vuota, 4, 0, adesso)
print(f"Statistiche su lista vuota: {statistiche_vuote}. Atteso: None.")

COLLEZIONE VUOTA
La collezione è vuota: nessun obiettivo da mostrare.
DASHBOARD MEDITACTIVE
Saldo monete: 0
Nessun dato disponibile: la collezione è vuota.
Nessun obiettivo trovato con questo ID o nome.

Statistiche su lista vuota: None. Atteso: None.


In [14]:
print("REGISTRAZIONI NON VALIDE")
collezione_prove = crea_collezione_iniziale()

esito = registrazione_nuovo_elemento(
    collezione_prove, "", "", "boh", -30, adesso
)
esito = registrazione_nuovo_elemento(
    collezione_prove, "Nuova attività", "Sonno", "boh", 30, adesso
)
esito = registrazione_nuovo_elemento(
    collezione_prove, "Nuova attività", "Sonno", "Giornaliera", -30, adesso
)
esito = registrazione_nuovo_elemento(
    collezione_prove, "Nuova attività", "Sonno", "Giornaliera", "trenta", adesso
)
esito = registrazione_nuovo_elemento(
    collezione_prove, "Nuova attività", "Sonno", "Giornaliera", True, adesso
)
print(f"Obiettivi dopo i tentativi: {len(collezione_prove)}. Atteso: 10.")

print("\nNOME DUPLICATO CON MAIUSCOLE DIVERSE")
esito = registrazione_nuovo_elemento(
    collezione_prove, "  ROUTINE PRE SONNO  ", "Sonno", "Mensile", 30, adesso
)
print(f"Inserimento riuscito: {esito}. Atteso: False.")

REGISTRAZIONI NON VALIDE
Errore: nome, tipo e frequenza non possono essere vuoti.
Errore: frequenza 'Boh' non valida.
Frequenze ammesse: Giornaliera, Settimanale, Mensile, Annuale.
Errore: la durata deve essere maggiore di zero.
Errore: la durata deve essere un numero intero.
Errore: la durata deve essere un numero intero.
Obiettivi dopo i tentativi: 10. Atteso: 10.

NOME DUPLICATO CON MAIUSCOLE DIVERSE
Errore: esiste già un obiettivo con questo nome.
Inserimento riuscito: False. Atteso: False.


In [15]:
print("FILTRO SENZA RISULTATI")
risultati_vuoti = filtro_avanzato(
    collezione_prove, adesso, nome="obiettivo inesistente"
)
visualizza_collezione(risultati_vuoti, adesso)

print("\nFILTRO CON CATEGORIA INESISTENTE")
risultati_tipo = filtro_avanzato(collezione_prove, adesso, tipo_cercato="meditaz")
print(f"Risultati: {len(risultati_tipo)}. Atteso: 0, ma con un errore esplicito")
print("invece di una lista vuota che sembra un risultato di ricerca.")

print("\nFILTRO NON VALIDO: DURATA COME TESTO")
risultati_errati = filtro_avanzato(collezione_prove, adesso, durata_massima="30")
print(f"Risultati: {len(risultati_errati)}. Atteso: 0, dopo il messaggio.")

print("\nIDENTIFICATIVO INESISTENTE")
saldo_prove = 10
saldo_prove = completa_obiettivo(collezione_prove, "999", saldo_prove, adesso)
print(f"Saldo dopo il tentativo: {saldo_prove}. Atteso: 10, invariato.")

FILTRO SENZA RISULTATI
La collezione è vuota: nessun obiettivo da mostrare.

FILTRO CON CATEGORIA INESISTENTE
Errore: categoria 'Meditaz' non valida.
Categorie ammesse: Meditazione, Movimento, Introspezione, Sonno.
Risultati: 0. Atteso: 0, ma con un errore esplicito
invece di una lista vuota che sembra un risultato di ricerca.

FILTRO NON VALIDO: DURATA COME TESTO
Errore: la durata massima deve essere un numero intero.
Risultati: 0. Atteso: 0, dopo il messaggio.

IDENTIFICATIVO INESISTENTE
Nessun obiettivo trovato con questo ID o nome.

Saldo dopo il tentativo: 10. Atteso: 10, invariato.


## 10. La frequenza alla prova del tempo

Qui la frequenza viene messa alla prova facendo scorrere le date. Poiché
l'istante è un parametro, basta passare un `adesso` diverso per vedere come si
comporta il programma domani o fra dieci giorni.

L'obiettivo `001` è giornaliero: due sessioni nello stesso giorno non sono
ammesse, una il giorno dopo sì. L'obiettivo `005` è mensile: dieci giorni non
bastano per ripeterlo.

Si noti che lo stato torna da solo a "da completare" quando il periodo scade.
Non c'è nessuna riga di codice che lo azzera, perché non c'è nessun campo da
azzerare: lo stato è il risultato di un confronto fra date.

In [16]:
collezione_periodo = crea_collezione_iniziale()
saldo_periodo = 0
mattina = datetime(2026, 9, 20, 7, 30)

print("OBIETTIVO GIORNALIERO — PRIMA SESSIONE")
saldo_periodo = completa_obiettivo(collezione_periodo, 1, saldo_periodo, mattina)

print("STESSO GIORNO, SEI ORE DOPO")
saldo_periodo = completa_obiettivo(
    collezione_periodo, 1, saldo_periodo, mattina + timedelta(hours=6)
)

print("IL GIORNO DOPO")
saldo_periodo = completa_obiettivo(
    collezione_periodo, 1, saldo_periodo, mattina + timedelta(days=1)
)

print("OBIETTIVO MENSILE — PRIMA SESSIONE")
saldo_periodo = completa_obiettivo(collezione_periodo, 5, saldo_periodo, mattina)

print("DIECI GIORNI DOPO: TROPPO PRESTO PER UN MENSILE")
saldo_periodo = completa_obiettivo(
    collezione_periodo, 5, saldo_periodo, mattina + timedelta(days=10)
)

sessioni_periodo = (
    collezione_periodo[0]["volte_completato"]
    + collezione_periodo[4]["volte_completato"]
)
print(f"Sessioni valide registrate: {sessioni_periodo}. Atteso: 3.")
print(f"Saldo: {saldo_periodo}. Atteso: 9 monete (2 + 2 + 5).")

OBIETTIVO GIORNALIERO — PRIMA SESSIONE
Completato: [001] Meditazione guidata mattutina
Sessioni registrate: 1
Monete guadagnate: 2
Saldo disponibile: 2 monete.

STESSO GIORNO, SEI ORE DOPO
Già completato: [001] Meditazione guidata mattutina
Frequenza giornaliera: una sessione al giorno.
Ultima sessione: 20/09/2026 07:30
Prossima sessione valida dal 21/09/2026 07:30.

IL GIORNO DOPO
Completato: [001] Meditazione guidata mattutina
Sessioni registrate: 2
Monete guadagnate: 2
Saldo disponibile: 4 monete.

OBIETTIVO MENSILE — PRIMA SESSIONE
Completato: [005] Sfida respirazione 4-7-8
Sessioni registrate: 1
Monete guadagnate: 5
Saldo disponibile: 9 monete.

DIECI GIORNI DOPO: TROPPO PRESTO PER UN MENSILE
Già completato: [005] Sfida respirazione 4-7-8
Frequenza mensile: una sessione ogni 30 giorni.
Ultima sessione: 20/09/2026 07:30
Prossima sessione valida dal 20/10/2026 07:30.

Sessioni valide registrate: 3. Atteso: 3.
Saldo: 9. Atteso: 9 monete (2 + 2 + 5).


## 11. Identificativi univoci e riscatto di più alberi

Due controlli su punti in cui è facile sbagliare.

**Gli identificativi dopo una rimozione.** Viene tolto l'obiettivo `004`, mentre
`010` resta al suo posto. Il nuovo inserimento deve ricevere `011`, anche se la
lista contiene soltanto nove elementi: un ID calcolato come `len() + 1` avrebbe
prodotto `010`, cioè un duplicato. `pop(3)` rimuove il quarto elemento, perché
gli indici partono da zero.

**Il riscatto di più alberi.** Una sessione da 510 minuti vale 102 monete. Il
ciclo `while` deve simulare due alberi, sottrarre 100 monete e lasciare un saldo
di 2: con un `if` al posto del `while` ne sarebbe stato piantato uno solo.

In [17]:
collezione_id = crea_collezione_iniziale()
rimosso = collezione_id.pop(3)
print(f"Obiettivo rimosso: {rimosso['ID']}")

esito = registrazione_nuovo_elemento(
    collezione_id, "Pausa dopo lo studio", "Meditazione", "Giornaliera", 10,
    adesso
)
print(f"Obiettivi nella lista: {len(collezione_id)}")
print(f"Nuovo ID: {collezione_id[-1]['ID']}. Atteso: 011.")

Obiettivo rimosso: 004
Obiettivo aggiunto: [011] Pausa dopo lo studio
Obiettivi nella lista: 10
Nuovo ID: 011. Atteso: 011.


In [18]:
collezione_alberi = []
saldo_alberi = 0

esito = registrazione_nuovo_elemento(
    collezione_alberi, "Giornata di movimento", "Movimento", "Annuale", 510,
    adesso
)
saldo_alberi = completa_obiettivo(collezione_alberi, 1, saldo_alberi, adesso)
print(f"Saldo residuo: {saldo_alberi}. Atteso: 2.")

Obiettivo aggiunto: [001] Giornata di movimento
Completato: [001] Giornata di movimento
Sessioni registrate: 1
Monete guadagnate: 102
Albero piantato nella simulazione! Costo: 50 monete.
Albero piantato nella simulazione! Costo: 50 monete.
Saldo disponibile: 2 monete.

Saldo residuo: 2. Atteso: 2.


## 12. Controlli finali

Sette controlli con `if/else` confrontano i risultati principali con i valori
attesi. Se un domani una modifica cambiasse un comportamento, il riepilogo
direbbe subito quale.

In [19]:
controlli_superati = 0

if len(collezione_obiettivi) == 12:
    print("OK: 12 obiettivi nella collezione principale.")
    controlli_superati += 1
else:
    print("DA CONTROLLARE: numero degli obiettivi.")

if saldo_monete == 14:
    print("OK: saldo finale di 14 monete, dopo un albero piantato.")
    controlli_superati += 1
else:
    print("DA CONTROLLARE: saldo delle monete.")

if riepilogo["sessioni_totali"] == 3:
    print("OK: tre sessioni valide, la quarta respinta dalla frequenza.")
    controlli_superati += 1
else:
    print("DA CONTROLLARE: conteggio delle sessioni.")

if riepilogo["tipo_piu_frequente"] == "Meditazione":
    print("OK: Meditazione è il tipo più frequente.")
    controlli_superati += 1
else:
    print("DA CONTROLLARE: tipo più frequente.")

if collezione_id[-1]["ID"] == "011":
    print("OK: identificativo corretto dopo la rimozione.")
    controlli_superati += 1
else:
    print("DA CONTROLLARE: generazione dell'identificativo.")

if saldo_alberi == 2:
    print("OK: saldo corretto dopo due alberi simulati.")
    controlli_superati += 1
else:
    print("DA CONTROLLARE: riscatto di più alberi.")

if saldo_periodo == 9:
    print("OK: la frequenza ha respinto le sessioni fuori periodo.")
    controlli_superati += 1
else:
    print("DA CONTROLLARE: regola della frequenza.")

print(f"\nControlli finali superati: {controlli_superati} su 7.")

OK: 12 obiettivi nella collezione principale.
OK: saldo finale di 14 monete, dopo un albero piantato.
OK: tre sessioni valide, la quarta respinta dalla frequenza.
OK: Meditazione è il tipo più frequente.
OK: identificativo corretto dopo la rimozione.
OK: saldo corretto dopo due alberi simulati.
OK: la frequenza ha respinto le sessioni fuori periodo.

Controlli finali superati: 7 su 7.


## Note e limiti

I dati vivono in memoria per la durata della sessione: non c'è un archivio su
disco, quindi riavviando il kernel si riparte dal dataset iniziale. Le sessioni,
le monete e gli alberi sono simulati; non viene effettuato nessun acquisto e non
viene piantato nessun albero reale.

La dimostrazione principale usa `datetime.now()`, quindi le date cambiano a ogni
esecuzione. Le prove sulla frequenza usano invece date fisse, così il loro esito
è sempre lo stesso.

I periodi sono contati in giorni interi: un mese vale trenta giorni e un anno
trecentosessantacinque, senza tener conto della lunghezza reale dei mesi né
degli anni bisestili. È una semplificazione accettabile per obiettivi di
benessere, dove conta la cadenza e non la data esatta.

I dizionari della collezione devono mantenere lo schema mostrato nella sezione
4; i nuovi elementi vanno inseriti con `registrazione_nuovo_elemento`, che è il
solo punto in cui lo schema viene creato.

Le docstring seguono la struttura della
[PEP 257](https://peps.python.org/pep-0257/): una frase iniziale che dice cosa
fa la funzione, una riga vuota, poi i parametri e il valore restituito.